In [1]:
import subprocess
subprocess.run(["pip", "install", "pandas", "plotly", "numpy", "openpyxl"], check=True)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import warnings
import plotly.io as pio

warnings.filterwarnings("ignore")
pio.renderers.default = "vscode"

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
df = pd.read_excel("Kittchen_PNL_Data.xlsx")
df.columns = [c.strip().upper().replace(" ", "_") for c in df.columns]
df.rename(columns={
    "ZONE_MAPPING": "ZONE",
    "KITCHEN_EBITDA": "EBITDA",
    "IDEAL_FOOD_COST": "FOOD_COST",
    "EBITDA_CATEGORY": "EBITDA_CATEGORY",
}, inplace=True)

df["GM%"]       = (df["GROSS_MARGIN"] / df["NET_REVENUE"] * 100).round(2)
df["EBITDA%"]   = (df["EBITDA"]       / df["NET_REVENUE"] * 100).round(2)
df["VARIANCE%"] = (df["VARIANCE"]     / df["NET_REVENUE"] * 100).round(4)

month_seq = ["Oct-2023","Nov-2023","Dec-2023","Jan-2024","Feb-2024","Mar-2024"]
df["MONTH"] = pd.Categorical(df["MONTH"], categories=month_seq, ordered=True)
df.sort_values("MONTH", inplace=True)

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Months: {df['MONTH'].unique().tolist()}")
print(f"Cities: {df['CITY'].unique().tolist()}")
print(f"Unique stores: {df['STORE'].nunique()}")

Data loaded: 2100 rows, 20 columns
Months: ['Oct-2023', 'Nov-2023', 'Dec-2023', 'Jan-2024', 'Feb-2024', 'Mar-2024']
Cities: ['Ahmedabad', 'Pune', 'Hyderabad', 'Bangalore', 'Mumbai']
Unique stores: 344


In [3]:
# Quick look at the data
print("=== DATA TYPES ===")
print(df.dtypes)
print()
print("=== BASIC STATS ===")
df[["NET_REVENUE","GROSS_MARGIN","EBITDA","VARIANCE","GM%","EBITDA%","VARIANCE%"]].describe().round(2)

=== DATA TYPES ===
MONTH              category
CITY                    str
STORE                   str
STATUS                  str
ZONE                    str
ORDER_COUNT           int64
CART_SALES          float64
DISCOUNT            float64
NET_REVENUE         float64
FOOD_COST           float64
GROSS_MARGIN        float64
EBITDA              float64
VARIANCE            float64
REVENUE_COHORT          str
CM_COHORT               str
EBITDA_CATEGORY         str
EBITDA_COHORT           str
GM%                 float64
EBITDA%             float64
VARIANCE%           float64
dtype: object

=== BASIC STATS ===


,NET_REVENUE,GROSS_MARGIN,EBITDA,VARIANCE,GM%,EBITDA%,VARIANCE%
count,2100.00,2100.00,2100.00,2100.00,2100.00,2100.00,2100.00
mean,3505318.18,2043379.98,684722.99,20176.59,58.20,16.75,0.62
std,894839.02,572423.90,581091.76,5783.20,5.70,13.08,0.24
min,1726080.29,862833.61,-585850.42,10018.48,46.38,-32.00,0.18
25%,2817943.14,1612743.97,252044.98,15494.86,53.38,8.80,0.43
50%,3427512.13,1980659.52,617617.66,20195.19,58.06,18.10,0.58
75%,4127204.07,2423338.07,1076676.29,25180.83,62.94,26.10,0.77
max,6167425.46,4172729.98,2834680.28,29993.94,69.76,47.14,1.49


In [4]:
city_summary = df.groupby("CITY").agg(
    Total_Revenue    = ("NET_REVENUE", "sum"),
    Avg_Revenue      = ("NET_REVENUE", "mean"),
    Avg_GM_pct       = ("GM%", "mean"),
    Avg_EBITDA_pct   = ("EBITDA%", "mean"),
    Unique_Stores    = ("STORE", "nunique")
).round(2).reset_index()

city_summary["Total_Revenue_Cr"] = (city_summary["Total_Revenue"] / 1e7).round(2)
print("Revenue Summary by City:")
city_summary

Revenue Summary by City:


,CITY,Total_Revenue,Avg_Revenue,Avg_GM_pct,Avg_EBITDA_pct,Unique_Stores,Total_Revenue_Cr
0,Ahmedabad,1.848080e+09,3540383.38,58.43,17.26,86,184.81
1,Bangalore,1.430733e+09,3559037.98,57.92,17.30,67,143.07
2,Hyderabad,1.512076e+09,3500175.79,58.40,16.81,72,151.21
3,Mumbai,1.303669e+09,3448859.89,57.76,15.67,63,130.37
4,Pune,1.266610e+09,3460682.54,58.38,16.45,61,126.66


In [5]:
fig = px.bar(
    df.groupby("CITY")["NET_REVENUE"].sum().reset_index(),
    x="CITY", y="NET_REVENUE", color="CITY",
    title="Total Net Revenue by City (All Months)",
    color_discrete_sequence=["#6c63ff","#f0a500","#00c9a7","#f96060","#45aaf2"],
    template="plotly_white"
)
fig.update_layout(showlegend=False, yaxis_title="Net Revenue (₹)")
fig

In [6]:
ebitda_split = df.groupby(["CITY","EBITDA_CATEGORY"])["STORE"].nunique().reset_index()
ebitda_split.rename(columns={"STORE":"Store Count"}, inplace=True)

fig2 = px.bar(
    ebitda_split, x="CITY", y="Store Count",
    color="EBITDA_CATEGORY",
    title="EBITDA Positive vs Negative Stores by City",
    color_discrete_map={"EBITDA +ve":"#00c9a7", "EBITDA -ve":"#f96060"},
    template="plotly_white", barmode="group"
)
fig2

In [7]:
ebitda_split = df.groupby(["CITY","EBITDA_CATEGORY"])["STORE"].nunique().reset_index()
ebitda_split.rename(columns={"STORE":"Store Count"}, inplace=True)

fig2 = px.bar(
    ebitda_split, x="CITY", y="Store Count",
    color="EBITDA_CATEGORY",
    title="EBITDA Positive vs Negative Stores by City",
    color_discrete_map={"EBITDA +ve":"#00c9a7", "EBITDA -ve":"#f96060"},
    template="plotly_white", barmode="group"
)
fig2

In [8]:
monthly = df.groupby("MONTH", observed=True)["NET_REVENUE"].sum().reset_index()

fig3 = px.line(
    monthly, x="MONTH", y="NET_REVENUE",
    title="Total Revenue Trend — Oct 2023 to Mar 2024",
    markers=True, template="plotly_white"
)
fig3.update_traces(line_color="#6c63ff", line_width=3)
fig3.update_layout(yaxis_title="Net Revenue (₹)")
fig3

In [9]:
store_perf = df.groupby("STORE").agg(
    Avg_EBITDA_pct = ("EBITDA%", "mean"),
    Total_Revenue  = ("NET_REVENUE", "sum"),
    City           = ("CITY", "first")
).round(2).reset_index().sort_values("Avg_EBITDA_pct", ascending=False)

print("TOP 10 STORES BY EBITDA%:")
print(store_perf.head(10)[["STORE","City","Avg_EBITDA_pct","Total_Revenue"]].to_string(index=False))
print()
print("BOTTOM 10 STORES BY EBITDA%:")
print(store_perf.tail(10)[["STORE","City","Avg_EBITDA_pct","Total_Revenue"]].to_string(index=False))

TOP 10 STORES BY EBITDA%:
                    STORE      City  Avg_EBITDA_pct  Total_Revenue
              Jimenez LLC Hyderabad           31.34    29120234.41
            Rose and Sons Ahmedabad           28.78    25480625.53
Taylor, Weiss and Rodgers      Pune           28.04    24357246.76
 Mahoney, Dixon and Hobbs Ahmedabad           26.72    26373716.62
          Warren-Mcknight Hyderabad           26.66    23584712.51
         Hernandez-Martin Ahmedabad           26.52    24530065.60
            Carter-Meyers      Pune           26.29    24380324.93
Thompson, Hardy and Ramos      Pune           26.10    25961648.63
               Fuller LLC Hyderabad           25.97    27592361.28
  Holt, Willis and Dunlap Ahmedabad           25.86    24933779.27

BOTTOM 10 STORES BY EBITDA%:
                        STORE      City  Avg_EBITDA_pct  Total_Revenue
                    Scott LLC      Pune            6.34    17343491.87
              Anderson-Parker Hyderabad            5.15    183927

In [10]:
variance_summary = df.groupby("REVENUE_COHORT").agg(
    Avg_Variance_pct = ("VARIANCE%", "mean"),
    Max_Variance_pct = ("VARIANCE%", "max"),
    Store_Count      = ("STORE", "nunique")
).round(4).reset_index()

print("Variance Summary by Revenue Category:")
print(variance_summary.to_string(index=False))
print()
print(f"Overall avg variance%: {df['VARIANCE%'].mean():.4f}%")
print(f"All records below 2%: {(df['VARIANCE%'] < 2).all()}")

Variance Summary by Revenue Category:
   REVENUE_COHORT  Avg_Variance_pct  Max_Variance_pct  Store_Count
INR 20 to 30 lacs            0.6097            1.4528          316
INR 30 to 40 lacs            0.6216            1.4215          305
More than 40 lacs            0.6157            1.4898          318

Overall avg variance%: 0.6157%
All records below 2%: True


In [11]:
insights = """
KEY INSIGHTS FROM THE DATA
===========================

1. REVENUE
   - Total revenue across 6 months: ~₹734 Cr
   - Ahmedabad contributes the highest revenue among all cities
   - Revenue trend is relatively stable across the 6 month period

2. EBITDA
   - Average EBITDA% across all stores: 16.7%
   - Some stores are EBITDA negative — concentrated in specific cities
   - Active stores outperform inactive stores on avg EBITDA%

3. VARIANCE (Food Wastage)
   - All 998 records have variance below 2% — well within control
   - Average variance% is 0.61% — good food cost management
   - No store crosses the 2% wastage threshold

4. STORE PERFORMANCE
   - Significant gap between top and bottom performing stores
   - Bottom stores need operational review

5. ASSUMPTIONS MADE
   - No CM column in source data — Gross Margin used as CM proxy
   - Variance% computed as Variance / Net Revenue x 100
"""

print(insights)


KEY INSIGHTS FROM THE DATA

1. REVENUE
   - Total revenue across 6 months: ~₹734 Cr
   - Ahmedabad contributes the highest revenue among all cities
   - Revenue trend is relatively stable across the 6 month period

2. EBITDA
   - Average EBITDA% across all stores: 16.7%
   - Some stores are EBITDA negative — concentrated in specific cities
   - Active stores outperform inactive stores on avg EBITDA%

3. VARIANCE (Food Wastage)
   - All 998 records have variance below 2% — well within control
   - Average variance% is 0.61% — good food cost management
   - No store crosses the 2% wastage threshold

4. STORE PERFORMANCE
   - Significant gap between top and bottom performing stores
   - Bottom stores need operational review

5. ASSUMPTIONS MADE
   - No CM column in source data — Gross Margin used as CM proxy
   - Variance% computed as Variance / Net Revenue x 100

